# Brain CT Segmentation — Colab runner

Before running: **Runtime -> Change runtime type -> T4 GPU** (or any GPU), otherwise training will run on CPU and be slow.

This notebook covers two ways to get the code onto Colab — use whichever applies to you right now:
- **Option A:** clone from your own GitHub repo (use this *after* you've pushed, see the deployment steps in chat)
- **Option B:** upload the project zip directly (works right now, before you've pushed anywhere)

Then it covers getting your dataset onto Colab, running a 2-epoch smoke test, then the full comparison sweep.

## 0. Check GPU is attached

In [ ]:
!nvidia-smi

## 1a. Option A — clone from GitHub
Replace the URL with your repo's URL once you've pushed it (see the GitHub deployment steps).

In [ ]:
!git clone https://github.com/<your-username>/<your-repo>.git
%cd <your-repo>

## 1b. Option B — upload the project zip instead
Run this cell, click **Choose Files**, and select `brain_seg_project.zip`. Skip 1a if you use this.

In [ ]:
from google.colab import files
uploaded = files.upload()  # select brain_seg_project.zip
!unzip -q -o brain_seg_project.zip
%cd brain_seg_project

## 2. Install dependencies

In [ ]:
!pip install -q -r requirements.txt

## 3. Get your dataset onto Colab

Pick ONE of these depending on where your ~100 images/masks currently live.

**3a. From Google Drive (recommended for repeated runs):**
```python
from google.colab import drive
drive.mount('/content/drive')
!cp -r "/content/drive/MyDrive/<path-to-your-images-folder>"/* data/images/
!cp -r "/content/drive/MyDrive/<path-to-your-masks-folder>"/*  data/masks/
```

**3b. Direct upload (fine for ~100 small images):**
```python
from google.colab import files
up = files.upload()  # select all CT image files
!mkdir -p data/images && mv *.png data/images/  # adjust extension if needed
```

**3c. From a zip of your dataset uploaded to Drive or via `files.upload()`:**
```python
!unzip -q -o your_dataset.zip -d data/
```

In [ ]:
# Sanity check: confirm files landed where the code expects them
!ls data/images | head
!ls data/masks | head
!echo "image count:" && ls data/images | wc -l
!echo "mask count:"  && ls data/masks  | wc -l

## 4. Smoke test — 2 epochs on one architecture
Confirms the whole pipeline runs on YOUR data before committing to the full sweep.

In [ ]:
%cd src
!python3 train.py --model unet_scratch --epochs 2 --batch_size 4

If that printed a `train_dice` / `val_dice` per epoch with no traceback, the pipeline works end to end on your data.

**Before the full run:** delete the smoke-test checkpoint so it isn't picked up as a "trained" result, but KEEP `results/split.json` — you want every architecture to share that same split.


In [ ]:
!rm -f ../results/unet_scratch_best.pt ../results/unet_scratch_history.json
%cd ..

## 5. Full run — train every architecture + compare

In [ ]:
!bash run_all.sh

## 6. View the comparison table

In [ ]:
import pandas as pd
df = pd.read_csv('results/comparison_table.csv')
df

In [ ]:
import matplotlib.pyplot as plt
df.plot(x='model', y=['dice', 'iou'], kind='bar', figsize=(8,4))
plt.ylabel('Score')
plt.title('Model comparison on held-out test set')
plt.tight_layout()
plt.show()

## 7. Download results back to your machine (optional)

In [ ]:
from google.colab import files
!zip -r results.zip results/
files.download('results.zip')